# Lab: Sessions & Compaction

**Module 02b — Context Engineering**

## Objectives

By the end of this lab you will be able to:

1. **Persist** a conversation as an append-only event log backed by SQLite.
2. **Enforce** deterministic ordering with a per-session lock.
3. **Implement** the sliding-window and recursive-summarization compaction strategies from the slides.
4. **Measure** the token-cost difference between raw, windowed, and compacted message lists.
5. **Combine** both strategies into a hybrid compactor.
6. **Extend** the compactor with guided summarization (`focus_topic`) and a `on_pre_compress` lifecycle hook — the Hermes Agent pattern.

## The Story

The slides argued that the messages list is a document the model completes — and every token in that document is billed on every turn. We're going to build the storage layer that lets us *keep* the full history while *sending* only what the model needs.

| Part | Topic |
|------|-------|
| 1 | A SQLite-backed `Session` (events + state, with per-session locking) |
| 2 | Walkthrough — append 12 turns of a support conversation |
| 3 | Compaction strategy A — sliding window |
| 4 | Compaction strategy B — recursive summarization |
| 5 | Exercise — write a hybrid compactor |
| 6 | Hermes pattern — guided compression with `focus_topic` + `on_pre_compress` hook |


## Setup

Install `litellm` and `python-dotenv`. SQLite ships with Python.

`OPENAI_API_KEY` (or any provider key supported by `litellm`) must be in your `.env`. We use a cheap model for summarization — that's the whole point of the slide's "summarize old turns with `deepseek/deepseek-v4-flash:free`" guidance.


In [ ]:
# !uv pip install litellm python-dotenv

import json
import os
import sqlite3
import threading
from contextlib import contextmanager
from pathlib import Path

import litellm
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("CHAT_MODEL", "deepseek/deepseek-v4-flash:free")
SUMMARY_MODEL = os.getenv("SUMMARY_MODEL", "deepseek/deepseek-v4-flash:free")

assert os.getenv("OPENROUTER_API_KEY"), (
    "Set OPENROUTER_API_KEY in .env"
)
print(f"chat model:    {CHAT_MODEL}")
print(f"summary model: {SUMMARY_MODEL}")


---
## Part 1: A SQLite-backed `Session`

The slide defined a minimal `Session(session_id, store)` with `events()` and `append(event)` against any KV store. We're going to use SQLite so the events survive a kernel restart — and so you can poke at the underlying table with `sqlite3 lab_sessions.db` from your terminal.

### Two things the slide insisted on

- **Append-only event log.** We never edit a row; new turns become new rows.
- **Deterministic ordering.** Concurrent turns must serialise. We use a per-session `threading.Lock` — `transactions` would be the production answer.


In [ ]:
DB_PATH = Path.cwd() / "lab_sessions.db"

# One lock per session_id, lazily created. Real systems use a DB transaction or
# a distributed lock; this is enough to demo the contract.
_session_locks: dict[str, threading.Lock] = {}
_locks_guard = threading.Lock()


def _lock_for(session_id: str) -> threading.Lock:
    with _locks_guard:
        if session_id not in _session_locks:
            _session_locks[session_id] = threading.Lock()
        return _session_locks[session_id]


@contextmanager
def _connect():
    conn = sqlite3.connect(DB_PATH)
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()


def _init_db() -> None:
    with _connect() as conn:
        # WAL lets readers and a writer coexist without blocking each other.
        conn.execute("PRAGMA journal_mode=WAL")
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS events (
                session_id TEXT NOT NULL,
                seq INTEGER NOT NULL,
                payload TEXT NOT NULL,
                PRIMARY KEY (session_id, seq)
            )
            """
        )


class Session:
    def __init__(self, session_id: str) -> None:
        self.session_id = session_id

    def events(self) -> list[dict]:
        with _connect() as conn:
            rows = conn.execute(
                "SELECT payload FROM events WHERE session_id = ? ORDER BY seq ASC",
                (self.session_id,),
            ).fetchall()
        return [json.loads(r[0]) for r in rows]

    def append(self, event: dict) -> None:
        # Single-statement INSERT computes the next seq atomically — no race
        # between the SELECT and the INSERT. The lock still serialises appends
        # within this process so callers see deterministic ordering.
        with _lock_for(self.session_id):
            with _connect() as conn:
                conn.execute(
                    """
                    INSERT INTO events(session_id, seq, payload)
                    VALUES (
                        ?,
                        (SELECT COALESCE(MAX(seq), -1) + 1 FROM events WHERE session_id = ?),
                        ?
                    )
                    """,
                    (
                        self.session_id,
                        self.session_id,
                        json.dumps(event, ensure_ascii=False),
                    ),
                )


_init_db()
print(f"events table ready at {DB_PATH}")


---
## Part 2: Walkthrough — a 12-turn support conversation

We'll append a small support dialogue to a fresh session, then read it back. The point isn't the dialogue — it's that the messages **round-trip** through SQLite in order.


In [ ]:
SESSION_ID = "demo-support-001"

# Fresh start for the demo: wipe any rows from a previous run.
with _connect() as conn:
    conn.execute("DELETE FROM events WHERE session_id = ?", (SESSION_ID,))

session = Session(SESSION_ID)

session.append({"role": "system", "content": "You are a polite SaaS support agent."})

fake_turns = [
    ("user",      "Hi, my dashboard is empty even though I uploaded a CSV yesterday."),
    ("assistant", "Sorry to hear that. Can you confirm the workspace name and the file size?"),
    ("user",      "Workspace acme-prod, file was 12MB."),
    ("assistant", "Got it. Let me check the ingestion log."),
    ("user",      "Sure, thanks."),
    ("assistant", "I can see the file landed but the schema inference failed because column 3 mixed dates and strings."),
    ("user",      "Ah, that column has 'N/A' for missing dates. Is that the issue?"),
    ("assistant", "Yes. Replace 'N/A' with an empty cell or pick a date placeholder and re-upload."),
    ("user",      "Will I lose the rows that already loaded?"),
    ("assistant", "No — failed schema inference means nothing was persisted. The re-upload is a clean load."),
    ("user",      "Perfect. I'll do it now."),
]
for role, content in fake_turns:
    session.append({"role": role, "content": content})

events = session.events()
print(f"persisted {len(events)} events")
for e in events[:3]:
    print(" ", e)
print("  ...")
for e in events[-2:]:
    print(" ", e)


> **What just happened.** Every `append` took the lock, computed the next sequence number, and wrote a row. `events()` reads them back in `seq` order — that's the deterministic-ordering guarantee from the slide's callout.


---
## Part 3: Compaction strategy A — sliding window

The slide's first compaction strategy is the simplest: keep the system message + the last *N* turns, drop the rest. Storage stays full; only the *payload sent to the model* shrinks.

We'll measure the win with `litellm.token_counter`, which counts tokens with the model's actual tokenizer.


In [ ]:
def trim_to_last_n(events: list[dict], n: int = 4) -> list[dict]:
    """Keep system messages + the last n non-system turns."""
    system = [e for e in events if e["role"] == "system"]
    rest = [e for e in events if e["role"] != "system"]
    return system + rest[-n:]


def count_tokens(messages: list[dict], model: str = CHAT_MODEL) -> int:
    return litellm.token_counter(model=model, messages=messages)


full = session.events()
windowed = trim_to_last_n(full, n=4)

full_tokens = count_tokens(full)
windowed_tokens = count_tokens(windowed)

print(f"full message list:    {len(full):>2} messages, {full_tokens:>4} tokens")
print(f"sliding window (n=4): {len(windowed):>2} messages, {windowed_tokens:>4} tokens")
print(f"reduction: {full_tokens - windowed_tokens} tokens ({100 * (1 - windowed_tokens / full_tokens):.0f}%)")


> **Trade-off.** The sliding window forgets the original problem statement once it falls off the back of the window. For a 12-turn debugging chat that's fine; for an advisor agent that tracks goals across days, it's catastrophic. That's why we need strategy B.


---
## Part 4: Compaction strategy B — recursive summarization

This is the slide's `compact(events, keep_recent=6)` verbatim, with the cheap-model summary call. We trade one extra LLM call for a permanently shorter message list.


In [ ]:
def compact(events: list[dict], keep_recent: int = 6, model: str = SUMMARY_MODEL) -> list[dict]:
    if len(events) <= keep_recent:
        return events

    old, recent = events[:-keep_recent], events[-keep_recent:]

    summary = litellm.completion(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Summarize this conversation in under 200 words. "
                    "Preserve facts, decisions, and open questions."
                ),
            },
            {"role": "user", "content": json.dumps(old, ensure_ascii=False)},
        ],
        temperature=0,
    ).choices[0].message.content

    return [{"role": "system", "content": f"Earlier conversation summary:\n{summary}"}] + recent


compacted = compact(session.events(), keep_recent=6)
compacted_tokens = count_tokens(compacted)

print(f"full:      {len(full):>2} messages, {full_tokens:>4} tokens")
print(f"windowed:  {len(windowed):>2} messages, {windowed_tokens:>4} tokens")
print(f"compacted: {len(compacted):>2} messages, {compacted_tokens:>4} tokens")
print()
print("--- summary the cheap model wrote ---")
print(compacted[0]["content"])


> **What it bought you.** The compacted list is a few hundred tokens longer than the raw window — that's the summary. But the *information density* is far higher: the earlier dialogue about the CSV mix-type bug is preserved, not dropped. Now the model can still reason about why the user is re-uploading even if the window walks past the original report.


### But did the summary keep what mattered?

Token reduction tells you what compaction *saved*. It says nothing about what it *lost* — and the lossy half is the dangerous one. The summarizer's prompt promised to "preserve facts, decisions, and open questions"; nothing so far has checked that it did.

Let's measure it. We enumerate the facts a support agent must not forget, then use the shared `eval_kit` judge to score whether each one survived the summary. (Same `LLMJudge` primitive you'll meet again in Lab 02 and Module 03.)


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "shared"))
from eval_kit.judge import LLMJudge

# The facts a support agent must NOT forget after compaction. These are the
# decisions and open questions the summarizer promised to preserve.
MUST_SURVIVE = [
    "The workspace is acme-prod and the uploaded file was about 12MB.",
    "Schema inference failed because a column mixed dates with the string 'N/A'.",
    "The fix is to replace 'N/A' with empty/placeholder dates and re-upload.",
    "Nothing was persisted on the failed load, so the re-upload is a clean load.",
]

summary_text = compacted[0]["content"]

fidelity_judge = LLMJudge(
    rubric=(
        "You are checking whether a conversation SUMMARY preserved a specific fact. "
        "Score 1 if the fact is stated or clearly derivable from the summary; "
        "score 0 if it was dropped or contradicted."
    ),
    scale_max=1,
)

kept = 0
for fact in MUST_SURVIVE:
    verdict = fidelity_judge.score(summary_text, context=f"Fact to verify: {fact}")
    kept += verdict.score
    print(f"[{'KEPT' if verdict.score else 'LOST'}] {fact}")

print(f"\nfidelity: {kept}/{len(MUST_SURVIVE)} key facts preserved")

# Treat fidelity as a gate, not just a metric. One judge miss is tolerable noise;
# losing more than that means the summary is too lossy to ship.
assert kept >= len(MUST_SURVIVE) - 1, (
    "Compaction dropped too much — tighten the summarizer prompt or raise keep_recent."
)


---
## Part 5: Exercise — hybrid compaction

The slide table promised a fourth strategy: **hybrid** = summary of old turns + verbatim of recent. You just built both pieces.

### Your task

Implement `compact_hybrid(events, keep_recent=4, summarize_older_than=8)`:

- If there are fewer than `summarize_older_than` events, return them unchanged (nothing to compact).
- Otherwise: keep the most recent `keep_recent` events verbatim, summarize the rest, prepend the summary.

Then re-run the token comparison and add a fourth row to the table.

In [ ]:
def compact_hybrid(
    events: list[dict],
    keep_recent: int = 4,
    summarize_older_than: int = 8,
    model: str = SUMMARY_MODEL,
) -> list[dict]:
    # TODO: implement the hybrid strategy described above.
    # Hint: this is `compact()` with an extra short-circuit at the top.
    raise NotImplementedError


# Once implemented:
# hybrid = compact_hybrid(session.events(), keep_recent=4, summarize_older_than=8)
# print(f"hybrid:    {len(hybrid):>2} messages, {count_tokens(hybrid):>4} tokens")

---
## Part 6: Hermes pattern — guided compression with a lifecycle hook

Open [`agent/conversation_compression.py`](https://github.com/NousResearch/hermes-agent/blob/main/agent/conversation_compression.py) in the Hermes Agent repo. Two design choices stand out beyond what we built in Part 5:

- **`focus_topic`** — a hint passed into the summarizer's system prompt so the summary preserves topic-relevant detail. Inspired by Claude Code's `/compact <focus>` command.
- **`on_pre_compress`** — a lifecycle hook fired *before* the summarizer runs. The hook receives the messages about to be compressed and can return structured insights that survive the compression (otherwise lost in the summary).

Hermes also protects the first *N* and last *M* messages from compression (the system prompt + the immediate working context). Our `keep_recent` does the last-M half — the first-N idea you could add as a follow-on exercise.

### Your task

Extend your hybrid compactor from Part 5 with these two parameters. Fill in the scaffold below; the **Solution** cell after it is collapsed — try it yourself first, then expand to compare. The demo cell after that runs your `compact_with_focus` against the support conversation.


In [ ]:
from typing import Callable, Optional

PreCompressHook = Callable[[list[dict]], Optional[str]]


def compact_with_focus(
    events: list[dict],
    keep_recent: int = 4,
    summarize_older_than: int = 8,
    *,
    focus_topic: Optional[str] = None,
    on_pre_compress: Optional[PreCompressHook] = None,
    model: str = SUMMARY_MODEL,
) -> list[dict]:
    """Hybrid compaction with guided summarization + a lifecycle hook.

    TODO: extend your Part-5 hybrid compactor with two new behaviors.
      1. Short-circuit: if len(events) < summarize_older_than, return events as-is.
      2. Split: old = events[:-keep_recent], recent = events[-keep_recent:].
      3. If on_pre_compress is given, call it with `old`; keep the returned note
         (a string, or None) — it should survive into the summary.
      4. Build the summarizer system prompt. If focus_topic is set, append a line
         telling the model to PRIORITIZE that topic and drop tangential detail.
      5. Summarize `old` with litellm (temperature=0). If you have a pre-compress
         note, prepend it to the summary body.
      6. Return [{"role": "system", "content": "Earlier conversation summary:\\n..."}]
         + recent.
    """
    raise NotImplementedError


> **Solution** below is collapsed — try the scaffold first, then expand to compare. Running the solution cell defines a working `compact_with_focus`, so the demo runs even if you skip the exercise.


In [ ]:
# @solution  — collapsed; expand to compare with your attempt above.
# Running it defines the working `compact_with_focus` so the demo below runs.
from typing import Callable, Optional

PreCompressHook = Callable[[list[dict]], Optional[str]]


def compact_with_focus(
    events: list[dict],
    keep_recent: int = 4,
    summarize_older_than: int = 8,
    *,
    focus_topic: Optional[str] = None,
    on_pre_compress: Optional[PreCompressHook] = None,
    model: str = SUMMARY_MODEL,
) -> list[dict]:
    """Hybrid compaction with guided summarization + a lifecycle hook.

    - focus_topic: a hint the summarizer treats as priority. Content related to
      it is preserved at higher fidelity; tangential detail is dropped first.
    - on_pre_compress: callback fired with the messages about to be compressed.
      Return a string to prepend to the summary, or None to skip.
    """
    if len(events) < summarize_older_than:
        return events

    old, recent = events[:-keep_recent], events[-keep_recent:]

    pre_compress_note: Optional[str] = None
    if on_pre_compress is not None:
        pre_compress_note = on_pre_compress(old)

    system_lines = [
        "Summarize this conversation in under 200 words.",
        "Preserve facts, decisions, and open questions.",
    ]
    if focus_topic:
        system_lines.append(
            f"PRIORITIZE detail about: {focus_topic}. Drop tangential content first."
        )
    system_prompt = " ".join(system_lines)

    summary = litellm.completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": json.dumps(old, ensure_ascii=False)},
        ],
        temperature=0,
    ).choices[0].message.content

    summary_body = summary
    if pre_compress_note:
        summary_body = f"[pre-compress note]\n{pre_compress_note}\n\n[summary]\n{summary}"

    return [
        {"role": "system", "content": f"Earlier conversation summary:\n{summary_body}"}
    ] + recent


In [ ]:
# Demo: pass a focus_topic and a hook that extracts user key points
# before the summarizer compresses the older messages.

def log_and_extract(messages: list[dict]) -> str:
    """on_pre_compress hook — runs before compression.

    Logs how many messages are about to be dropped, then returns a short
    structured note that survives the summary (prepended to it).
    """
    print(f"[hook] about to compress {len(messages)} messages")
    user_lines = [m["content"] for m in messages if m["role"] == "user"]
    return "User key points: " + " | ".join(user_lines[:3])


focused = compact_with_focus(
    session.events(),
    keep_recent=4,
    summarize_older_than=8,
    focus_topic="the CSV column-type bug and the re-upload plan",
    on_pre_compress=log_and_extract,
)

print()
print(f"focused:   {len(focused):>2} messages, {count_tokens(focused):>4} tokens")
print()
print("--- focused summary block ---")
print(focused[0]["content"])


### Read the real thing

The production pattern lives in two files in [`NousResearch/hermes-agent`](https://github.com/NousResearch/hermes-agent):

- [`agent/context_engine.py`](https://github.com/NousResearch/hermes-agent/blob/main/agent/context_engine.py) — when to compress (token thresholds, protected first-N / last-M windows, manual `/compress` override).
- [`agent/conversation_compression.py`](https://github.com/NousResearch/hermes-agent/blob/main/agent/conversation_compression.py) — how to compress (auxiliary summarizer LLM, `focus_topic` plumbing, `on_pre_compress` extraction).

What you just built is the same shape: a hybrid summary + verbatim split, a hint that steers the summarizer, and a hook that lets the agent rescue structured insights *before* messages are dropped. The Hermes implementation adds token-aware scheduling, fallback models, and a `/compress` command — all worth a read.


---
## Reflection

### Key Takeaways

| Concept | What you learned |
|---|---|
| **Storage vs. payload** | The slide's rule: keep the full history persisted; only compact what you *send*. SQLite stays fat; the messages list stays slim. |
| **Per-session lock** | Concurrent appends would interleave events and corrupt the conversation. Production systems use DB transactions; the lock is the same idea, scoped to one process. |
| **Sliding window** | Cheapest, but drops early context. Right for short bursty chats. |
| **Recursive summarization** | One extra LLM call buys you "the model still remembers what we were doing". Right for long advisor agents. |
| **Hybrid** | Best quality, most complex. Use after you've measured that the window alone is losing too much. |

### Connection to Module 03

The session is the **scaffold** that the agent loop appends events to. When you reach Module 03's newsroom (Lab 02), every specialist's tool call and reply will pass through the same append-only events idea — just sliced into spans by `@observe`. Same primitive, different presentation.

### Bonus

Run `compact()` inside an `asyncio.create_task(...)` so the summarization happens between turns without blocking the next user message. The slide flagged this explicitly: *"Run this asynchronously between turns. The next turn just reads the cached compacted version."*
